# DINOv2 self-supervised training on embryo time-lapse `.npy` (1 GPU)

**Dữ liệu**: file JSON chứa danh sách đường dẫn `.npy`; mỗi file có shape `(11, 600, 600, 1)` uint8.
Chỉ dùng frame **1 → 8** (bỏ 0, 9, 10). Mỗi frame là **1 sample**, ảnh xám được nhân thành 3 kênh RGB
để dùng đúng chuẩn hoá ImageNet và weight pretrain của DINOv2.

**Mặc định**: ViT-L/14 khởi tạo từ weight DINOv2 chính thức (LVD-142M), batch 32, bf16 — vừa với 1 GPU 48GB
(đã tính cho RTX PRO 5000 Blackwell).

**Chế độ 1 GPU** (`train.use_fsdp=False`): không dùng FSDP / NCCL, mixed precision bằng `torch.autocast` (bf16),
attention bằng `scaled_dot_product_attention` của PyTorch → chạy với **torch mới (vd. 2.11 + cu128)**, **không cần xFormers**.

Môi trường: Linux, `torch` + `torchvision` (bản hỗ trợ GPU của bạn), `omegaconf`, `fvcore`, `iopath`, `matplotlib`.

Các bước:
1. Cấu hình đường dẫn & hyper-parameters
2. Tạo dataset + kiểm tra dữ liệu, xem augmentation
3. Khởi tạo config / process group 1-GPU
4. Tạo model, nạp weight pretrain
5. Train (tự resume nếu `OUTPUT_DIR` đã có checkpoint)
6. Export teacher backbone & thử trích feature

In [ ]:
import os
import sys

# xFormers không cần thiết. Mặc định tắt (bản xFormers cho torch 2.11 / Blackwell chưa được kiểm chứng).
# Phải đặt TRƯỚC khi import dinov2.
USE_XFORMERS = False
if not USE_XFORMERS:
    os.environ["XFORMERS_DISABLED"] = "1"

# Thêm thư mục gốc của repo vào sys.path (notebook nằm trong <repo>/notebooks)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(REPO_ROOT, "dinov2")):
    REPO_ROOT = os.getcwd()  # trường hợp chạy notebook ngay tại gốc repo
assert os.path.isdir(os.path.join(REPO_ROOT, "dinov2")), "Hãy sửa REPO_ROOT trỏ tới thư mục chứa package dinov2"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from dinov2.data.datasets import NpyFramesDataset
from dinov2.data import DataAugmentationDINO
from dinov2.train.ssl_meta_arch import SSLMetaArch
from dinov2.train.train import do_train
from dinov2.train import notebook_utils as nbu

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory / 1024**3:.1f} GB")

## 1. Cấu hình

- `JSON_PATH`: file JSON. Có thể là **list** đường dẫn, hoặc **dict** dạng `{"train": [...], "val": [...]}` → khi đó đặt `JSON_SPLIT="train"`
  (để `None` sẽ gộp tất cả các list).
- `DATA_ROOT`: tiền tố ghép vào trước đường dẫn tương đối trong JSON
  (VD JSON chứa `data/nas-server/...` và dữ liệu nằm ở `/data/nas-server/...` → `DATA_ROOT="/"`).
  Để `""` nghĩa là tương đối với thư mục làm việc hiện tại (`os.getcwd()`).
- `INIT_WEIGHTS`: `"official"` = tải weight DINOv2 khớp với kiến trúc; hoặc đường dẫn file `.pth`; hoặc `None` = train từ đầu.

In [ ]:
# ---------------- Dữ liệu ----------------
JSON_PATH = "/path/to/npy_list.json"   # <-- SỬA
JSON_SPLIT = None                       # hoặc "train" nếu JSON là dict
DATA_ROOT = ""                          # <-- SỬA nếu đường dẫn trong JSON là tương đối
FRAME_INDICES = (1, 2, 3, 4, 5, 6, 7, 8)  # bỏ frame 0, 9, 10

# ---------------- Model ----------------
ARCH = "vit_large"           # vit_small | vit_base | vit_large  (giant không khuyến nghị với 1 GPU)
PATCH_SIZE = 14              # weight chính thức của DINOv2 đều là patch 14
NUM_REGISTER_TOKENS = 0      # 0 cho dinov2_vitl14_pretrain.pth, 4 cho dinov2_vitl14_reg4_pretrain.pth
# "official" = tự tải https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth
# hoặc đường dẫn file .pth "backbone only" đã tải về; None = train từ đầu
INIT_WEIGHTS = "official"

MODEL_NAME = f"dinov2_{ARCH.replace('_', '')[:4]}{PATCH_SIZE}" + (f"_reg{NUM_REGISTER_TOKENS}" if NUM_REGISTER_TOKENS else "")
OUTPUT_DIR = os.path.join(REPO_ROOT, "outputs", f"{MODEL_NAME}_embryo")

# ---------------- Train ----------------
BATCH_SIZE = 32              # ViT-L/14: bắt đầu 32, xem "max mem" trong log rồi tăng (40-48) nếu còn dư. ViT-B/14: 64
NUM_WORKERS = 8
EPOCHS = 30                  # 1 epoch = 1 lượt qua toàn bộ frame (số file x 8)
WARMUP_EPOCHS = 3
# base_lr được scale: lr = base_lr * sqrt(BATCH_SIZE / 1024)
#   - tiếp tục từ weight pretrain: 5e-4  (-> lr ~8.8e-5 với batch 32)
#   - train từ đầu:               4e-3  (giá trị mặc định của DINOv2)
BASE_LR = 5e-4 if INIT_WEIGHTS else 4e-3
SAVE_TEACHER_EVERY_EPOCHS = 1
SEED = 0

## 2. Dataset & kiểm tra dữ liệu

In [ ]:
dataset = NpyFramesDataset(
    root=JSON_PATH,
    split=JSON_SPLIT,
    data_root=DATA_ROOT,
    frame_indices=FRAME_INDICES,
)
print(f"{len(dataset.files):,d} file .npy  ->  {len(dataset):,d} sample (frame)")
print("Ví dụ:", dataset.files[:3])

In [ ]:
# Kiểm tra nhanh một số file: tồn tại, shape, dtype, đủ số frame
import random

N_CHECK = 50
rng = random.Random(0)
to_check = rng.sample(dataset.files, min(N_CHECK, len(dataset.files)))
max_frame = max(FRAME_INDICES)
problems = []
for path in to_check:
    if not os.path.isfile(path):
        problems.append((path, "không tồn tại"))
        continue
    arr = np.load(path, mmap_mode="r")
    if arr.dtype != np.uint8 or arr.ndim not in (3, 4) or arr.shape[0] <= max_frame:
        problems.append((path, f"shape={arr.shape} dtype={arr.dtype}"))
print(f"Đã kiểm tra {len(to_check)} file, lỗi: {len(problems)}")
for p in problems[:10]:
    print("  ", p)
if problems and not os.path.isfile(problems[0][0]):
    print("\n=> Có thể DATA_ROOT chưa đúng. cwd =", os.getcwd())

In [ ]:
# Xem 8 frame của 1 file (đã chuyển sang RGB)
file_idx = 0
n = len(FRAME_INDICES)
fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.4))
for j in range(n):
    img = dataset.get_image(file_idx * n + j)
    axes[j].imshow(img)
    axes[j].set_title(f"frame {FRAME_INDICES[j]}")
    axes[j].axis("off")
print("mode:", img.mode, "size:", img.size)
plt.tight_layout(); plt.show()

In [ ]:
# Xem các crop augmentation mà DINOv2 sẽ tạo ra (2 global 224px + 8 local 98px)
GLOBAL_CROPS_SIZE, LOCAL_CROPS_SIZE, LOCAL_CROPS_NUMBER = 224, 98, 8   # 224 = 16*14, 98 = 7*14

aug = DataAugmentationDINO(
    global_crops_scale=(0.32, 1.0),
    local_crops_scale=(0.05, 0.32),
    local_crops_number=LOCAL_CROPS_NUMBER,
    global_crops_size=GLOBAL_CROPS_SIZE,
    local_crops_size=LOCAL_CROPS_SIZE,
)
out = aug(dataset.get_image(0))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
crops = out["global_crops"] + out["local_crops"]
fig, axes = plt.subplots(1, len(crops), figsize=(1.8 * len(crops), 2))
for ax, c in zip(axes, crops):
    ax.imshow((c * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(f"{c.shape[-1]}px"); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Config cho 1 GPU 48GB

Các lựa chọn chính:
- `student.img_size=518`: positional embedding giữ đúng kích thước của weight chính thức (37×37), được nội suy khi chạy ở 224/98
  → không mất thông tin pretrain, và backbone export ra có cùng định dạng với `torch.hub` DINOv2.
- `train.use_fsdp=False`: train 1 GPU kiểu PyTorch thường (không FSDP / NCCL) — tương thích torch mới.
- `compute_precision.amp_dtype="bf16"`: autocast bf16 (GPU Blackwell/Ampere+ hỗ trợ), ổn định hơn fp16, không cần grad scaler.
  Đổi `"fp16"` nếu muốn (khi đó grad scaler tự bật).
- `block_chunks=0`: tên key khớp trực tiếp với weight chính thức.
- `OFFICIAL_EPOCH_LENGTH = len(dataset) // BATCH_SIZE` để "epoch" đúng nghĩa 1 lượt dữ liệu.
- **Nếu OOM**: giảm `BATCH_SIZE` (32 → 24) hoặc `LOCAL_CROPS_NUMBER` (8 → 6); cân nhắc `head_n_prototypes=32768`.

In [ ]:
epoch_length = max(1, len(dataset) // BATCH_SIZE)
warmup_teacher_temp_epochs = min(10, max(1, EPOCHS // 3))

overrides = {
    "MODEL": {"WEIGHTS": ""},
    "train": {
        "batch_size_per_gpu": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "OFFICIAL_EPOCH_LENGTH": epoch_length,
        "dataset_path": f"NpyFrames:root={JSON_PATH}",  # chỉ để ghi vào config.yaml; dataset thật được truyền trực tiếp
        "seed": SEED,
        "use_fsdp": False,          # 1 GPU: không FSDP / NCCL
    },
    "compute_precision": {"amp_dtype": "bf16"},
    "student": {
        "arch": ARCH,
        "patch_size": PATCH_SIZE,
        "img_size": 518,
        "num_register_tokens": NUM_REGISTER_TOKENS,
        "drop_path_rate": 0.1 if ARCH in ("vit_small", "vit_base") else 0.3,
        "layerscale": 1.0e-05,
        "ffn_layer": "mlp",
        "block_chunks": 0,
        "pretrained_weights": "",   # weight được nạp ở bước 4 (hỗ trợ đúng định dạng checkpoint chính thức)
    },
    "teacher": {
        "momentum_teacher": 0.994,
        "final_momentum_teacher": 1.0,
        "warmup_teacher_temp": 0.04,
        "teacher_temp": 0.07,
        "warmup_teacher_temp_epochs": warmup_teacher_temp_epochs,
    },
    "dino": {"head_n_prototypes": 65536, "koleo_loss_weight": 0.1},
    "ibot": {"head_n_prototypes": 65536},
    "optim": {
        "epochs": EPOCHS,
        "warmup_epochs": WARMUP_EPOCHS,
        "base_lr": BASE_LR,
        "min_lr": 1.0e-06,
        "weight_decay": 0.04,
        "weight_decay_end": 0.2,
        "layerwise_decay": 0.9,
        "patch_embed_lr_mult": 0.2,
        "clip_grad": 3.0,
        "freeze_last_layer_epochs": 1,
    },
    "crops": {
        "global_crops_size": GLOBAL_CROPS_SIZE,
        "local_crops_size": LOCAL_CROPS_SIZE,
        "local_crops_number": LOCAL_CROPS_NUMBER,
        "global_crops_scale": [0.32, 1.0],
        "local_crops_scale": [0.05, 0.32],
    },
    "evaluation": {"eval_period_iterations": SAVE_TEACHER_EVERY_EPOCHS * epoch_length},
}

assert GLOBAL_CROPS_SIZE % PATCH_SIZE == 0 and LOCAL_CROPS_SIZE % PATCH_SIZE == 0
assert WARMUP_EPOCHS < EPOCHS

cfg = nbu.make_cfg(overrides, OUTPUT_DIR)
cfg = nbu.setup_single_gpu(cfg, seed=SEED)   # logging + seed + lr scaling + lưu config.yaml

total_iters = cfg.optim.epochs * cfg.train.OFFICIAL_EPOCH_LENGTH
print(f"lr sau scaling: {cfg.optim.lr:.2e}")
print(f"iter/epoch: {epoch_length:,d} | tổng iter: {total_iters:,d} | checkpoint đầy đủ mỗi {3 * epoch_length:,d} iter")

## 4. Model + weight pretrain
Weight được nạp vào **student backbone**; `prepare_for_distributed_training()` sẽ copy student → teacher (ở chế độ 1 GPU không bọc FSDP).
Hai head DINO/iBOT luôn khởi tạo ngẫu nhiên (checkpoint chính thức không có head).
Nếu `OUTPUT_DIR` đã có checkpoint training, bước train sẽ **resume** và ghi đè các weight này.

In [ ]:
model = SSLMetaArch(cfg).to(torch.device("cuda"))

if INIT_WEIGHTS:
    weights_src = nbu.official_weights_url(cfg) if INIT_WEIGHTS == "official" else INIT_WEIGHTS
    print("Nạp weight:", weights_src)
    nbu.load_backbone_weights(model.student.backbone, weights_src)

model.prepare_for_distributed_training()
n_params = sum(p.numel() for p in model.student.backbone.parameters())
print(f"student backbone: {n_params / 1e6:.1f}M params")

## 5. Train
- Log (loss, lr, thời gian, bộ nhớ GPU `max mem`) in ra mỗi 10 iter, đồng thời lưu ở `OUTPUT_DIR/log.txt` và `training_metrics.json`.
- Checkpoint đầy đủ (để resume): `OUTPUT_DIR/model_*.pth` + file `last_checkpoint`, mỗi 3 epoch (giữ 3 bản gần nhất).
- Teacher checkpoint (để dùng sau này): `OUTPUT_DIR/eval/training_<iter>/teacher_checkpoint.pth`.
- **Dừng giữa chừng / resume**: interrupt kernel → **Restart kernel** → chạy lại toàn bộ notebook; training tiếp tục từ checkpoint đầy đủ gần nhất.

In [ ]:
do_train(cfg, model, resume=True, dataset=dataset)

In [ ]:
# Vẽ đường loss từ training_metrics.json
import json

metrics_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
rows = [json.loads(l) for l in open(metrics_path) if l.strip()]
it = [r["iteration"] for r in rows]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for k in ("total_loss", "dino_local_crops_loss", "dino_global_crops_loss", "koleo_loss", "ibot_loss"):
    if k in rows[0]:
        axes[0].plot(it, [r[k] for r in rows], label=k)
axes[0].legend(); axes[0].set_xlabel("iteration"); axes[0].set_title("loss")
axes[1].plot(it, [r["lr"] for r in rows]); axes[1].set_title("lr"); axes[1].set_xlabel("iteration")
plt.tight_layout(); plt.show()

## 6. Export teacher backbone & trích feature
File export là state dict của backbone, cùng định dạng key với weight DINOv2 chính thức.

In [ ]:
ckpts = nbu.find_teacher_checkpoints(OUTPUT_DIR)
print("Teacher checkpoints:", [i for i, _ in ckpts])
last_iter, last_ckpt = ckpts[-1]
export_path = nbu.export_teacher_backbone(last_ckpt, os.path.join(OUTPUT_DIR, f"backbone_teacher_{last_iter}.pth"))

backbone = nbu.build_backbone_from_cfg(cfg, weights=export_path).cuda()

# Trích feature cho vài frame (resize về 224, chuẩn hoá ImageNet)
from torchvision import transforms as T
eval_tf = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
x = torch.stack([eval_tf(dataset.get_image(i)) for i in range(8)]).cuda()
with torch.no_grad():
    feats = backbone.forward_features(x)
print("CLS:", feats["x_norm_clstoken"].shape, "| patch tokens:", feats["x_norm_patchtokens"].shape)

In [ ]:
# (Tuỳ chọn) Dùng lại ở chỗ khác qua torch.hub mà không cần config:
# hub_model = torch.hub.load(REPO_ROOT, MODEL_NAME, source="local", pretrained=False)
# hub_model.load_state_dict(torch.load(export_path, map_location="cpu"))